# Exercise 3: PostgreSQL Database Setup

In this exercise, I'm setting up a local PostgreSQL database to store the M5 data. I'm testing two different ways to load the data (Pandas and Polars) to see which one is faster. I've also decided to use a dedicated schema called `m5` to keep things organized.

In [ ]:
import os
import time

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

from tis3il.m5_utils import PATHS, PROCESSED, require_file

TABLES = {
    "calendar": PATHS.calendar,
    "sell_prices": PATHS.prices,
    "sales_train_validation": PATHS.sales_train,
    "sales_test_evaluation": PATHS.sales_test,
}

load_dotenv()
user = os.environ["POSTGRES_USER"]
password = os.environ["POSTGRES_PASSWORD"]
host = os.getenv("POSTGRES_HOST", "localhost")
port = os.getenv("POSTGRES_PORT", "5432")
db = os.getenv("POSTGRES_DB", "tsdb")
schema = os.getenv("POSTGRES_SCHEMA", "m5")
url = f"postgresql+psycopg://{user}:{password}@{host}:{port}/{db}"
engine = create_engine(url)

with engine.begin() as conn:
    conn.execute(text(f'CREATE SCHEMA IF NOT EXISTS "{schema}"'))

timings = []
for name, path in TABLES.items():
    started = time.perf_counter()
    df = pd.read_parquet(require_file(path))
    df.to_sql(name, engine, schema=schema, if_exists="replace", index=False, method="multi", chunksize=10_000)
    timings.append({"method": "pandas", "table": name, "rows": len(df), "seconds": round(time.perf_counter() - started, 3)})

import polars as pl
for name, path in TABLES.items():
    started = time.perf_counter()
    df = pl.read_parquet(require_file(path))
    df.write_database(table_name=f"{schema}.{name}_pl", connection=url, if_table_exists="replace")
    timings.append({"method": "polars", "table": f"{name}_pl", "rows": df.height, "seconds": round(time.perf_counter() - started, 3)})

timing_df = pd.DataFrame(timings)
timing_df.to_csv(PROCESSED / "exercise3_io_timings.csv", index=False)
timing_df